### Consumer Pricing

Transactional records w/ premise and items

- https://data.gov.my/data-catalogue/pricecatcher
- https://data.gov.my/data-catalogue/lookup_item
- https://data.gov.my/data-catalogue/lookup_premise


In [ ]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install sklearn
%pip install scipy
%pip install pyarrow
%pip install fastparquet

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import fastparquet
import pyarrow
import pyarrow.parquet as pq
import pyarrow.csv as pcsv
import pyarrow.ipc as pipc
import pyarrow.json as pjson
import pyarrow.orc as porc

In [2]:
# Source URLs from the catalog metadata pages
TRANSACTION_URL = "https://storage.data.gov.my/pricecatcher/pricecatcher_2026-02.parquet"
ITEM_URL = "https://storage.data.gov.my/pricecatcher/lookup_item.parquet"
PREMISE_URL = "https://storage.data.gov.my/pricecatcher/lookup_premise.parquet"


def load_dataset(parquet_url: str) -> pd.DataFrame:
    """Load parquet when available; fallback to CSV if parquet engine is missing."""
    try:
        return pd.read_parquet(parquet_url)
    except Exception:
        csv_url = parquet_url.replace(".parquet", ".csv")
        return pd.read_csv(csv_url)


# Load datasets
transaction_df = load_dataset(TRANSACTION_URL)
item_df = load_dataset(ITEM_URL)
premise_df = load_dataset(PREMISE_URL)

if "date" in transaction_df.columns:
    transaction_df["date"] = pd.to_datetime(
        transaction_df["date"], errors="coerce")

display(pd.DataFrame([
    {"table": "transaction_df",
        "rows": transaction_df.shape[0], "cols": transaction_df.shape[1]},
    {"table": "item_df", "rows": item_df.shape[0], "cols": item_df.shape[1]},
    {"table": "premise_df",
        "rows": premise_df.shape[0], "cols": premise_df.shape[1]},
]))

display(transaction_df.head())

,table,rows,cols
0,transaction_df,1216577,4
1,item_df,757,5
2,premise_df,3838,6


,date,premise_code,item_code,price
0,2026-02-01,3,2,10.6
1,2026-02-01,3,88,14.9
2,2026-02-01,3,92,14.2
3,2026-02-01,3,94,18.0
4,2026-02-01,3,95,7.6


In [3]:
# Metadata (from data.gov.my catalog pages)
metadata = {
    "pricecatcher": {
        "description": "Transactional price records.",
        "variables": ["date", "premise_code", "item_code", "price"],
        "url": TRANSACTION_URL,
        "join_key": ["premise_code", "item_code"],
    },
    "lookup_item": {
        "description": "Item lookup table.",
        "variables": ["item_code", "item_name", "unit", "item_group", "item_category"],
        "url": ITEM_URL,
        "join_key": ["item_code"],
    },
    "lookup_premise": {
        "description": "Premise lookup table.",
        "variables": ["premise_code", "premise", "address", "premise_type", "state", "district"],
        "url": PREMISE_URL,
        "join_key": ["premise_code"],
    },
}

In [4]:
# Join to one raw table (left joins preserve all transactions)
joined_df = (
    transaction_df
    .merge(item_df, on="item_code", how="left", validate="many_to_one")
    .merge(premise_df, on="premise_code", how="left", validate="many_to_one")
)

display(pd.DataFrame(
    [{"table": "joined_df", "rows": joined_df.shape[0], "cols": joined_df.shape[1]}]))
display(joined_df.head())

,table,rows,cols
0,joined_df,1216577,13


,date,premise_code,item_code,price,item,unit,item_group,item_category,premise,address,premise_type,state,district
0,2026-02-01,3,2,10.6,AYAM BERSIH - SUPER,1kg,BARANGAN SEGAR,AYAM,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah
1,2026-02-01,3,88,14.9,IKAN KELI (ANTARA 2 HINGGA 5 EKOR SEKILOGRAM),1kg,BARANGAN SEGAR,IKAN DARAT,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah
2,2026-02-01,3,92,14.2,CILI HIJAU,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah
3,2026-02-01,3,94,18.0,CILI MERAH - MINYAK,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah
4,2026-02-01,3,95,7.6,HALIA BASAH (TUA),1kg,BARANGAN SEGAR,SAYUR-SAYURAN,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah


In [5]:
# Save raw joined dataset (before cleaning/normalization)
os.makedirs("data", exist_ok=True)
raw_output_parquet = "data/pricecatcher_joined_raw.parquet"
raw_output_csv = "data/pricecatcher_joined_raw.csv"

try:
    joined_df.to_parquet(raw_output_parquet, index=False)
    save_status = f"Saved raw joined dataset to: {raw_output_parquet}"
except Exception:
    joined_df.to_csv(raw_output_csv, index=False)
    save_status = f"Parquet engine unavailable; saved CSV instead: {raw_output_csv}"

# Quick EDA snapshot
display(save_status)
eda_summary = {
    "rows": len(joined_df),
    "columns": joined_df.shape[1],
    "duplicates": int(joined_df.duplicated().sum()),
}
display(pd.DataFrame([eda_summary]))

display(joined_df.dtypes.to_frame("dtype"))
display((joined_df.isna().mean() *
        100).sort_values(ascending=False).to_frame("missing_pct"))

'Saved raw joined dataset to: data/pricecatcher_joined_raw.parquet'

,rows,columns,duplicates
0,1216577,13,0


,dtype
date,datetime64[ns]
premise_code,int64
item_code,int64
price,float64
item,object
unit,object
item_group,object
item_category,object
premise,object
address,object


,missing_pct
item,11.582004
unit,11.582004
item_group,11.582004
item_category,11.582004
date,0.000000
premise_code,0.000000
item_code,0.000000
price,0.000000
premise,0.000000
address,0.000000


### Filter for core analysis facet: `item_category`

For category-based analysis, rows without `item_category` are excluded. The missing share is about **11%**, so removing these records still preserves roughly **89%** of observations, which remains large enough to maintain strong coverage for downstream EDA.


In [6]:
# Drop rows with missing item_category (main analysis facet)
analysis_df = joined_df.dropna(subset=["item_category"]).copy()

rows_before = len(joined_df)
rows_after = len(analysis_df)
rows_dropped = rows_before - rows_after
pct_dropped = (rows_dropped / rows_before) * 100
pct_kept = 100 - pct_dropped

filter_summary = pd.DataFrame([
    {
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_dropped": rows_dropped,
        "pct_dropped": pct_dropped,
        "pct_kept": pct_kept,
    }
])
display(filter_summary)

# EDA on transaction date range
date_range_summary = pd.DataFrame([
    {
        "earliest_date": analysis_df["date"].min(),
        "latest_date": analysis_df["date"].max(),
    }
])
display(date_range_summary)

# Missing values check after filtering
display((analysis_df.isna().mean() *
        100).sort_values(ascending=False).to_frame("missing_pct"))

,rows_before,rows_after,rows_dropped,pct_dropped,pct_kept
0,1216577,1075673,140904,11.582004,88.417996


,earliest_date,latest_date
0,2026-02-01,2026-02-24


,missing_pct
date,0.0
premise_code,0.0
item_code,0.0
price,0.0
item,0.0
unit,0.0
item_group,0.0
item_category,0.0
premise,0.0
address,0.0


### Cardinality review and column-reduction decisions

We assess cardinality for `item`, `unit`, and `price` to check whether these fields are highly variant or close to invariant for analysis purposes.

Planned reductions for the working analysis dataset:

- Drop `item_code` and `premise_code` because they are technical join keys.
- Drop `address` because it is too granular for the current level of analysis.

This keeps semantic business fields while removing keys and overly granular location text.


In [7]:
# Cardinality analysis for selected columns
cardinality_cols = ["item", "unit", "price"]
cardinality_summary = pd.DataFrame(
    {
        "n_unique": analysis_df[cardinality_cols].nunique(dropna=True),
        "unique_ratio_pct": (analysis_df[cardinality_cols].nunique(dropna=True) / len(analysis_df)) * 100,
    }
).sort_values("n_unique", ascending=False)

display(cardinality_summary)
display(analysis_df["item"].value_counts().head(10).to_frame("count"))
display(analysis_df["unit"].value_counts().head(10).to_frame("count"))

# Build reduced analysis dataset by dropping keys and overly granular location text
drop_cols = ["item_code", "premise_code", "address"]
analysis_reduced_df = analysis_df.drop(columns=drop_cols).copy()

display(pd.DataFrame([{"reduced_rows": analysis_reduced_df.shape[0],
        "reduced_cols": analysis_reduced_df.shape[1]}]))
display(pd.DataFrame({"dropped_columns": drop_cols}))
display(pd.DataFrame({"remaining_columns": list(analysis_reduced_df.columns)}))

,n_unique,unique_ratio_pct
price,2152,0.200061
item,268,0.024915
unit,59,0.005485


,count
item,
HALIA BASAH (TUA),22987
BAWANG BESAR KUNING/HOLLAND,22850
BAWANG PUTIH IMPORT (CHINA),22513
LOBAK MERAH,21972
TIMUN,21835
TOMATO,21373
KUBIS BUNGA (CAULIFLOWER),20525
UBI KENTANG IMPORT (CHINA),20444
KUNYIT HIDUP,19958


,count
unit,
1kg,767755
250 g,38852
30 biji,36099
5 kg,17554
2 kg,17090
500 g,16135
850g,15900
340 g,14502
425 g,10273


,reduced_rows,reduced_cols
0,1075673,10


,dropped_columns
0,item_code
1,premise_code
2,address


,remaining_columns
0,date
1,price
2,item
3,unit
4,item_group
5,item_category
6,premise
7,premise_type
8,state
9,district


### Unit normalization and derived price metrics

Cleaning and feature-engineering steps (single integrated cell):

1. Normalize `unit` by trimming leading/trailing whitespace and removing internal spaces (for example, `340 g` -> `340g`, `5 kg` -> `5kg`).
2. Parse quantity + metric from the normalized unit string.
3. Standardize mass units to kilograms in `unit_in_kg`:
   - `kg` -> unchanged quantity
   - `g` -> quantity / 1000
4. Standardize volume units to liters in `unit_in_liter`:
   - `liter` -> unchanged quantity
   - `ml` -> quantity / 1000
5. Support complex patterns in the same pass (for example `5x79g`, `+-450g`, `2x200ml`).
6. Create `price_per_kg` and `price_per_liter` where respective standardized units are available.
7. Report remaining non-conforming units after all parsing rules.


In [8]:
# Normalize unit strings and derive standardized unit columns + price ratios
analysis_cleaned_df = analysis_reduced_df.copy()

# Step 1: normalize whitespace and casing in unit
analysis_cleaned_df["unit"] = (
    analysis_cleaned_df["unit"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
    .str.lower()
)

units = analysis_cleaned_df["unit"]

# Step 2: parse simple pattern (e.g. 340g, 1kg, 200ml, 1liter)
simple_parts = units.str.extract(
    r"^(?P<value>\d+(?:\.\d+)?)(?P<metric>kg|g|liter|ml)$"
)
simple_value = pd.to_numeric(simple_parts["value"], errors="coerce")
simple_metric = simple_parts["metric"]

# Step 3: parse multipack pattern (e.g. 5x79g, 2x200ml)
multi_parts = units.str.extract(
    r"^(?P<count>\d+)x(?P<size>\d+(?:\.\d+)?)(?P<metric>kg|g|liter|ml)$"
)
multi_count = pd.to_numeric(multi_parts["count"], errors="coerce")
multi_size = pd.to_numeric(multi_parts["size"], errors="coerce")
multi_metric = multi_parts["metric"]

# Step 4: parse signed/approximate prefix pattern (e.g. +-450g)
signed_parts = units.str.extract(
    r"^[+\-]+(?P<value>\d+(?:\.\d+)?)(?P<metric>kg|g|liter|ml)$"
)
signed_value = pd.to_numeric(signed_parts["value"], errors="coerce")
signed_metric = signed_parts["metric"]

# Build normalized metric and quantity in one pass
normalized_value = simple_value.copy()
normalized_metric = simple_metric.copy()

multi_mask = normalized_value.isna() & multi_count.notna() & multi_size.notna()
normalized_value = normalized_value.where(
    ~multi_mask, multi_count * multi_size)
normalized_metric = normalized_metric.where(~multi_mask, multi_metric)

signed_mask = normalized_value.isna() & signed_value.notna()
normalized_value = normalized_value.where(~signed_mask, signed_value)
normalized_metric = normalized_metric.where(~signed_mask, signed_metric)

# Step 5: convert to standardized mass/volume columns
analysis_cleaned_df["unit_in_kg"] = normalized_value * \
    normalized_metric.map({"kg": 1.0, "g": 0.001})
analysis_cleaned_df["unit_in_liter"] = normalized_value * \
    normalized_metric.map({"liter": 1.0, "ml": 0.001})

# Step 6: derive price-per-standard-unit columns
analysis_cleaned_df["price_per_kg"] = np.where(
    analysis_cleaned_df["unit_in_kg"].notna() & (
        analysis_cleaned_df["unit_in_kg"] > 0),
    analysis_cleaned_df["price"] / analysis_cleaned_df["unit_in_kg"],
    np.nan,
)
analysis_cleaned_df["price_per_liter"] = np.where(
    analysis_cleaned_df["unit_in_liter"].notna() & (
        analysis_cleaned_df["unit_in_liter"] > 0),
    analysis_cleaned_df["price"] / analysis_cleaned_df["unit_in_liter"],
    np.nan,
)

# Step 7: assign unit_type
analysis_cleaned_df["unit_type"] = np.select(
    [
        analysis_cleaned_df["unit_in_kg"].notna(),
        analysis_cleaned_df["unit_in_liter"].notna(),
    ],
    ["mass", "volume"],
    default="count_or_other",
)

# Step 8: diagnostics and optional exclusion of non-conforming units
unparsed_mask = (analysis_cleaned_df["unit_in_kg"].isna(
) & analysis_cleaned_df["unit_in_liter"].isna())
analysis_modeled_df = analysis_cleaned_df.loc[~unparsed_mask].copy()

conversion_summary = pd.DataFrame([
    {
        "rows_total": len(analysis_cleaned_df),
        "rows_kept_after_drop": len(analysis_modeled_df),
        "rows_dropped_unparsed": int(unparsed_mask.sum()),
        "pct_dropped_unparsed": (unparsed_mask.mean() * 100),
        "null_unit_in_kg": int(analysis_cleaned_df["unit_in_kg"].isna().sum()),
        "null_unit_in_liter": int(analysis_cleaned_df["unit_in_liter"].isna().sum()),
    }
])
display(conversion_summary)

display(analysis_cleaned_df["unit_type"].value_counts().to_frame("count"))

display(
    analysis_cleaned_df[["unit", "unit_type", "unit_in_kg",
                         "unit_in_liter", "price", "price_per_kg", "price_per_liter"]]
    .drop_duplicates(subset=["unit"])
    .head(20)
)

display(
    analysis_cleaned_df.loc[unparsed_mask, "unit"]
    .value_counts()
    .head(15)
    .to_frame("count")
)

display((analysis_modeled_df.isna().mean() *
        100).sort_values(ascending=False).to_frame("missing_pct_after_drop"))

,rows_total,rows_kept_after_drop,rows_dropped_unparsed,pct_dropped_unparsed,null_unit_in_kg,null_unit_in_liter
0,1075673,1032594,43079,4.004842,98470,1020282


,count
unit_type,
mass,977203
volume,55391
count_or_other,43079


,unit,unit_type,unit_in_kg,unit_in_liter,price,price_per_kg,price_per_liter
0,1kg,mass,1.000,NaN,10.60,10.600000,NaN
13,30biji,count_or_other,NaN,NaN,12.95,NaN,NaN
159792,425g,mass,0.425,NaN,10.50,24.705882,NaN
159793,325g,mass,0.325,NaN,4.20,12.923077,NaN
159794,5kg,mass,5.000,NaN,30.50,6.100000,NaN
159795,250g,mass,0.250,NaN,6.20,24.800000,NaN
159798,500g,mass,0.500,NaN,4.30,8.600000,NaN
159802,340g,mass,0.340,NaN,4.50,13.235294,NaN
159804,2kg,mass,2.000,NaN,13.30,6.650000,NaN
159841,5x79g,mass,0.395,NaN,5.40,13.670886,NaN


,count
unit,
30biji,36099
1batang,3295
100beg,2000
1biji,1685


,missing_pct_after_drop
unit_in_liter,94.635743
price_per_liter,94.635743
unit_in_kg,5.364257
price_per_kg,5.364257
date,0.000000
price,0.000000
item,0.000000
unit,0.000000
item_group,0.000000
item_category,0.000000


In [12]:
# Save cleaned modeled dataset
os.makedirs("data/cleaned", exist_ok=True)
cleaned_output_parquet = "data/cleaned/pricecatcher_modeled_cleaned.parquet"
cleaned_output_csv = "data/cleaned/pricecatcher_modeled_cleaned.csv"

try:
    analysis_modeled_df.to_parquet(cleaned_output_parquet, index=False)
    cleaned_save_status = f"Saved cleaned dataset to: {cleaned_output_parquet}"
except Exception:
    analysis_modeled_df.to_csv(cleaned_output_csv, index=False)
    cleaned_save_status = f"Parquet engine unavailable; saved cleaned dataset to: {cleaned_output_csv}"

display(cleaned_save_status)
display(pd.DataFrame(
    [{"rows": analysis_modeled_df.shape[0], "columns": analysis_modeled_df.shape[1]}]))

'Saved cleaned dataset to: data/cleaned/pricecatcher_modeled_cleaned.parquet'

,rows,columns
0,1032594,15
